# 04 - Risk Prediction Baselines and LazyPredict

## Problem framing
We model `Recovery_Status` as a multi-class prediction task and derive high-risk borrowers from the written-off class.

## Why LazyPredict?
- Fast baseline exploration
- Useful for rapid model sanity checks
- Lower control than hand-crafted pipelines


## Definition
Risk prediction estimates recovery outcomes and supports high-risk early warning decisions.

## Theory
This section explains the statistical or ML theory behind the technique and why it is useful in credit recovery operations.

## Mathematical Intuition
We translate the idea into score/probability/ranking logic so each metric can be interpreted quantitatively.

## Financial Intuition
We connect the method to borrower affordability, delinquency severity, collateral protection, and expected recoverable cashflows.

## Business Impact
We explain what this enables for collection managers, risk teams, and executives.

## Real-World Example
Two borrowers with similar outstanding balances can receive different actions if their written-off probability diverges.

## Visual Explanation
Charts in this notebook show how model/segment behavior changes across borrower groups.

## Code Explanation
Every code cell below is paired with interpretation so beginners can connect implementation details to business outcomes.

## Interpretation of Results
We summarize what changed, why it matters, and how to act on it.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.loan_recovery import (
    DATA_PATH,
    FIGURES_DIR,
    MODELS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    LoanDataLoader,
    FeatureEngineer,
    LoanEDA,
    BorrowerSegmenter,
    ModelTrainer,
    ModelEvaluator,
    RecoveryStrategyEngine,
    ModelExplainer,
    DashboardBuilder,
    LazyPredictBenchmark,
    PyCaretWorkflow,
    FLAMLOptimizer,
    SmartLoanRecoveryPipeline,
    load_model,
)

sns.set_theme(style="whitegrid")


In [2]:
def ensure_pipeline_artifacts() -> None:
    required = [
        TABLES_DIR / "manual_model_leaderboard.csv",
        TABLES_DIR / "feature_enriched_data.csv",
        MODELS_DIR / "best_manual_model.joblib",
    ]
    if not all(path.exists() for path in required):
        pipeline = SmartLoanRecoveryPipeline(data_path=DATA_PATH, random_state=42)
        pipeline.run()

ensure_pipeline_artifacts()


In [3]:
df = LoanDataLoader(DATA_PATH).load()
fe = FeatureEngineer()
enriched = fe.engineer(df)
split = fe.train_test_split(enriched, target_col="Recovery_Status", drop_cols=["Borrower_ID"])

trainer = ModelTrainer(random_state=42)
manual = trainer.train_baselines(split.x_train, split.y_train, split.x_test, split.y_test)
display(manual.leaderboard)


/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Smart Loan Recovery System/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Smart Loan Recovery System/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr,pr_auc_high_risk
0,SVM,0.39,0.3979,0.39,0.3929,0.5072,0.1991
1,Extra Trees,0.41,0.3397,0.41,0.3713,0.4723,0.1804
2,CatBoost,0.40,0.3263,0.40,0.3589,0.4436,0.1831
3,Random Forest,0.37,0.3105,0.37,0.3357,0.4359,0.1627
4,Gradient Boosting,0.34,0.3201,0.34,0.3258,0.4565,0.1879
5,AdaBoost,0.34,0.3027,0.34,0.3202,0.4651,0.1853
6,Logistic Regression,0.31,0.3342,0.31,0.3194,0.4885,0.1800
7,XGBoost,0.35,0.2928,0.35,0.3187,0.4643,0.1950
8,KNN,0.36,0.2937,0.36,0.3185,0.4423,0.1639
9,LightGBM,0.28,0.2797,0.28,0.2768,0.4383,0.1859


In [4]:
x_train_lazy = pd.get_dummies(split.x_train)
x_test_lazy = pd.get_dummies(split.x_test)
x_train_lazy, x_test_lazy = x_train_lazy.align(x_test_lazy, axis=1, join="left", fill_value=0)

lazy = LazyPredictBenchmark(random_state=42)
lazy_df = lazy.run(x_train_lazy, split.y_train, x_test_lazy, split.y_test)
display(lazy_df)
display(lazy.required_model_snapshot())


,model,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken
0,SVC,0.43,0.352869,NaN,0.384536,0.347848,0.43,0.030488
1,LogisticRegression,0.40,0.337446,0.502228,0.380612,0.367797,0.40,0.034869
2,AdaBoostClassifier,0.38,0.313187,0.473378,0.356174,0.336806,0.38,0.165851
3,RandomForestClassifier,0.38,0.309524,0.447729,0.339609,0.311387,0.38,0.372851
4,KNeighborsClassifier,0.37,0.303419,0.479022,0.336000,0.309695,0.37,0.041449
5,ExtraTreesClassifier,0.34,0.278388,0.459209,0.310013,0.285467,0.34,0.195464
6,CatBoostClassifier,0.31,0.257021,0.407933,0.287566,0.270818,0.31,5.256740
7,XGBClassifier,0.29,0.249534,0.436691,0.281877,0.275390,0.29,0.519846


,model,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken
0,SVC,0.43,0.352869,NaN,0.384536,0.347848,0.43,0.030488
1,LogisticRegression,0.40,0.337446,0.502228,0.380612,0.367797,0.40,0.034869
2,AdaBoostClassifier,0.38,0.313187,0.473378,0.356174,0.336806,0.38,0.165851
3,RandomForestClassifier,0.38,0.309524,0.447729,0.339609,0.311387,0.38,0.372851
4,KNeighborsClassifier,0.37,0.303419,0.479022,0.336000,0.309695,0.37,0.041449
5,ExtraTreesClassifier,0.34,0.278388,0.459209,0.310013,0.285467,0.34,0.195464
6,CatBoostClassifier,0.31,0.257021,0.407933,0.287566,0.270818,0.31,5.256740
7,XGBClassifier,0.29,0.249534,0.436691,0.281877,0.275390,0.29,0.519846


In [5]:
manual_top = manual.leaderboard[["model", "accuracy", "f1_weighted", "roc_auc_ovr"]].copy()
manual_top["source"] = "Manual"

lazy_top = lazy_df[["model", "Accuracy", "F1 Score"]].copy()
lazy_top.columns = ["model", "accuracy", "f1_weighted"]
lazy_top["roc_auc_ovr"] = np.nan
lazy_top["source"] = "LazyPredict"

comparison = pd.concat([manual_top, lazy_top], ignore_index=True)
display(comparison.sort_values(["source", "f1_weighted"], ascending=[True, False]))


,model,accuracy,f1_weighted,roc_auc_ovr,source
10,SVC,0.43,0.384536,NaN,LazyPredict
11,LogisticRegression,0.40,0.380612,NaN,LazyPredict
12,AdaBoostClassifier,0.38,0.356174,NaN,LazyPredict
13,RandomForestClassifier,0.38,0.339609,NaN,LazyPredict
14,KNeighborsClassifier,0.37,0.336000,NaN,LazyPredict
15,ExtraTreesClassifier,0.34,0.310013,NaN,LazyPredict
16,CatBoostClassifier,0.31,0.287566,NaN,LazyPredict
17,XGBClassifier,0.29,0.281877,NaN,LazyPredict
0,SVM,0.39,0.392900,0.5072,Manual
1,Extra Trees,0.41,0.371300,0.4723,Manual


## Tradeoff Summary
- **Manual ML**: highest control and deployment clarity.
- **LazyPredict**: strongest for quick benchmark discovery.
- **Weakness**: less configurable preprocessing and less business-specific tuning.
